# Vietnamese TTS Studio on Colab

Notebook nay dung cho Colab de:

1. Clone hoac pull repository tu GitHub.
2. Cai dependencies cho webapp TTS.
3. Chay Flask app tren Colab.
4. Mo Cloudflare tunnel de lay URL public.

Mac dinh notebook da tro san den repo hien tai:
`https://github.com/sin0235/text2speech.git`.


In [14]:
from pathlib import Path

REPO_URL = "https://github.com/sin0235/text2speech.git"
BRANCH = "main"
WORKSPACE = Path("/content/workspace")
PROJECT_DIR = WORKSPACE / Path(REPO_URL).stem.replace('.git', '')
PORT = 8386

INSTALL_F5 = True
INSTALL_VIRA = True
VIRA_AUTO_DOWNLOAD = True
DEFAULT_ENGINE = "vira"
HF_TOKEN = ""

TORCH_VERSION = "2.6.0"
TORCHVISION_VERSION = "0.21.0"
TORCHAUDIO_VERSION = "2.6.0"
PYTORCH_GPU_INDEX = "https://download.pytorch.org/whl/cu124"
PYTORCH_CPU_INDEX = "https://download.pytorch.org/whl/cpu"

VIRA_EXTRA_PACKAGES = [
    "spaces",
    "lmdeploy",
    "librosa",
    "einops",
    "onnxruntime-gpu",
    "soundfile",
    "soe-vinorm",
    "huggingface_hub",
    "omegaconf",
    "gradio",
    "ncodec @ https://github.com/ysharma3501/FastBiCodec/archive/refs/heads/master.zip",
    "fastaudiosr @ https://github.com/ysharma3501/FlashSR/archive/refs/heads/master.zip",
]

print({
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "project_dir": str(PROJECT_DIR),
    "port": PORT,
    "install_f5": INSTALL_F5,
    "install_vira": INSTALL_VIRA,
    "vira_auto_download": VIRA_AUTO_DOWNLOAD,
    "default_engine": DEFAULT_ENGINE,
    "torch": TORCH_VERSION,
    "torchvision": TORCHVISION_VERSION,
    "torchaudio": TORCHAUDIO_VERSION,
})


{'repo_url': 'https://github.com/sin0235/text2speech.git', 'branch': 'main', 'project_dir': '/content/workspace/text2speech', 'port': 8386, 'install_f5': True, 'install_vira': True, 'vira_auto_download': True, 'default_engine': 'vira', 'torch': '2.6.0', 'torchvision': '0.21.0', 'torchaudio': '2.6.0'}


In [15]:
import subprocess

WORKSPACE.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--short"], check=True)
print(f"Project ready at: {PROJECT_DIR}")


Project ready at: /content/workspace/text2speech


In [16]:
import shutil
import subprocess
import sys

subprocess.run(["apt-get", "update", "-y"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg", "libsndfile1", "curl"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)

has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
pytorch_index = PYTORCH_GPU_INDEX if has_gpu else PYTORCH_CPU_INDEX
print({"has_gpu": has_gpu, "pytorch_index": pytorch_index})

# Fix cho loi: libtorchaudio.so undefined symbol torch_list_push_back
# Nguyen nhan la torch va torchaudio lech binary ABI.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall",
    "--no-cache-dir",
    f"torch=={TORCH_VERSION}",
    f"torchvision=={TORCHVISION_VERSION}",
    f"torchaudio=={TORCHAUDIO_VERSION}",
    "--index-url", pytorch_index,
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"), "huggingface_hub"], check=True)

if INSTALL_F5:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "f5-tts"], check=True)

if INSTALL_VIRA:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "--extra-index-url", "https://huggingface.github.io/lmdeploy-wheel-index/",
        *VIRA_EXTRA_PACKAGES,
    ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "git+https://github.com/iamdinhthuan/Vira-tts.git"], check=True)

import torch
import torchaudio
print({
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "cuda_available": torch.cuda.is_available(),
})
print("Dependencies installed with matched torch/torchaudio versions.")


{'has_gpu': True, 'pytorch_index': 'https://download.pytorch.org/whl/cu124'}
{'torch': '2.6.0+cu124', 'torchaudio': '2.6.0+cu124', 'cuda_available': True}
Dependencies installed with matched torch/torchaudio versions.


In [17]:
import os

RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV["TTS_DEFAULT_ENGINE"] = DEFAULT_ENGINE
RUNTIME_ENV["VIRA_AUTO_DOWNLOAD"] = "1" if VIRA_AUTO_DOWNLOAD else "0"
RUNTIME_ENV["VIRA_MODEL_ID"] = "dolly-vn/Vira-TTS"

if HF_TOKEN.strip():
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Hugging Face token configured.")
else:
    print("HF_TOKEN is empty. Public downloads only.")

print({k: RUNTIME_ENV[k] for k in ["TTS_DEFAULT_ENGINE", "VIRA_AUTO_DOWNLOAD", "VIRA_MODEL_ID"]})


HF_TOKEN is empty. Public downloads only.
{'TTS_DEFAULT_ENGINE': 'vira', 'VIRA_AUTO_DOWNLOAD': '1', 'VIRA_MODEL_ID': 'dolly-vn/Vira-TTS'}


In [23]:
import os
import signal
import subprocess
import time

APP_LOG = PROJECT_DIR / "webapp_colab.log"
APP_PID = PROJECT_DIR / "webapp_colab.pid"

if APP_PID.exists():
    try:
        os.kill(int(APP_PID.read_text().strip()), signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

subprocess.run(["bash", "-lc", f"fuser -k {PORT}/tcp || true"], check=False)

with APP_LOG.open("w", encoding="utf-8") as log_file:
    app_proc = subprocess.Popen(
        ["python", "webapp/app.py"],
        cwd=str(PROJECT_DIR),
        env=RUNTIME_ENV,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

APP_PID.write_text(str(app_proc.pid), encoding="utf-8")
time.sleep(8)
print(f"Flask PID: {app_proc.pid}")
print(APP_LOG.read_text(encoding='utf-8', errors='ignore')[-4000:])


Flask PID: 17788
 * Serving Flask app 'app'
 * Debug mode: on
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8386
 * Running on http://172.28.0.12:8386
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 116-216-784



In [24]:
from pathlib import Path
import stat
import subprocess
import urllib.request

CLOUDFLARED_BIN = Path("/usr/local/bin/cloudflared")
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not CLOUDFLARED_BIN.exists():
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)

subprocess.run([str(CLOUDFLARED_BIN), "--version"], check=True)
print(f"cloudflared ready at {CLOUDFLARED_BIN}")


cloudflared ready at /usr/local/bin/cloudflared


In [25]:
import os
import re
import signal
import subprocess
import time
from IPython.display import HTML, display

TUNNEL_LOG = PROJECT_DIR / "cloudflared.log"
TUNNEL_PID = PROJECT_DIR / "cloudflared.pid"

if TUNNEL_PID.exists():
    try:
        os.kill(int(TUNNEL_PID.read_text().strip()), signal.SIGTERM)
        time.sleep(2)
    except Exception:
        pass

with TUNNEL_LOG.open("w", encoding="utf-8") as log_file:
    tunnel_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
        cwd=str(PROJECT_DIR),
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

TUNNEL_PID.write_text(str(tunnel_proc.pid), encoding="utf-8")
pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
public_url = None

for _ in range(90):
    time.sleep(1)
    log_text = TUNNEL_LOG.read_text(encoding='utf-8', errors='ignore') if TUNNEL_LOG.exists() else ""
    match = pattern.search(log_text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Khong tim thay trycloudflare URL. Kiem tra cloudflared.log.")

print("Public URL:", public_url)
display(HTML(f'<a href="{public_url}" target="_blank">Open TTS webapp</a>'))


Public URL: https://tiles-acquisition-again-robots.trycloudflare.com


In [26]:
print("=== Flask log ===")
print((PROJECT_DIR / "webapp_colab.log").read_text(encoding="utf-8", errors="ignore")[-4000:])
print("\n=== Cloudflare log ===")
print((PROJECT_DIR / "cloudflared.log").read_text(encoding="utf-8", errors="ignore")[-4000:])


=== Flask log ===
 * Serving Flask app 'app'
 * Debug mode: on
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8386
 * Running on http://172.28.0.12:8386
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 116-216-784


=== Cloudflare log ===
2026-04-03T15:16:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-03T15:16:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-03

In [22]:
import os
import signal
from pathlib import Path

for pid_file in [PROJECT_DIR / "webapp_colab.pid", PROJECT_DIR / "cloudflared.pid"]:
    if pid_file.exists():
        try:
            os.kill(int(pid_file.read_text().strip()), signal.SIGTERM)
            print(f"Stopped PID from {pid_file.name}")
        except Exception as exc:
            print(f"Could not stop {pid_file.name}: {exc}")


Stopped PID from webapp_colab.pid
Stopped PID from cloudflared.pid
